# ChaoSSM — Phase 13: generality across chaotic map families

## Why this notebook exists

Every result so far uses ONE chaotic generator: the coupled logistic-fold map. A reviewer can
reasonably ask whether the two headline findings -- (a) end-to-end training suppresses chaos on
copy, and (b) a frozen, random-input reservoir benefits from edge-of-chaos dynamics on forecast
-- are properties of chaos in general, or artifacts of that one map's construction.

This notebook re-runs the two decisive comparisons across FOUR chaotic map families, holding the
surrounding architecture (coupling W_mix, input gate, gain, fold-bounding, calibration procedure)
fixed and swapping only the elementwise local nonlinearity:

- **logistic**: r*h*(1-h)            (the main paper's map; included as the reference point)
- **tent**:     piecewise-linear tent map, natively bounded, no gain-driven onset
- **cubic**:    r*h*(1-h^2)-style Chebyshev-family cubic map (period-doubling route to chaos)
- **sine**:     r*sin(pi*h)          (the sine/logistic-equivalent map, different functional form)

All four are calibrated to the SAME target Lyapunov exponents (edge ~0-, deep ~+0.3) using the
same gamma-bisection procedure validated in Phase 9/10, so cross-map comparisons are regime-matched
rather than confounded by "which map happens to be more chaotic at its default parameters."

## Design

Two focused sub-experiments, run for all four map families:

**(A) Suppression under end-to-end training (generalizes the main paper's core finding).**
Gated cell, trained end-to-end on copy. Measures conditional lambda of the trained model.
Decision: does conditional lambda stay negative (suppression holds) across all four map families,
or only for the logistic map?

**(B) Benefit under the canonical frozen reservoir protocol (generalizes the Phase 10 finding).**
Recurrence AND input frozen random, readout-only training, on forecast (chaos-congenial task).
Compares chaos-edge vs a matched linear reservoir within each map family.
Decision: does the edge-of-chaos reservoir beat the linear reservoir across all four families
(as in Phase 10 for the logistic map), or is the benefit specific to one map's structure?

3 seeds per condition (this is a generality check across systems, not a new precision claim;
the main paper's 5-seed results are not superseded by this notebook).

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase13_map_generality"):
    try:
        from google.colab import drive; drive.mount("/content/drive",force_remount=False)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"./{project}_results/{phase}"); print(f"[info] not colab ({type(e).__name__})")
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase13_map_generality


## Configuration

In [2]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64; n_layers:int=1; mlp_mult:int=2
    steps_A:int=3000     # experiment A (suppression, gated, end-to-end)
    steps_B:int=2500     # experiment B (frozen reservoir, readout-only -- cheap)
    batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; lr:float=1e-3
    seeds:int=3
    eval_n:int=1200; eval_seed:int=12345
    copy_len:int=12; vocab:int=16
    fc_length:int=64; fc_bins:int=32
    lambda_target:dict=field(default_factory=lambda: dict(edge=-0.02, deep=0.30))
    lambda_tol:float=0.06; calib_steps:int=150; calib_warm:int=50
    benettin_warmup:int=80; benettin_steps:int=300; benettin_traj:int=10
    map_families:tuple=("logistic","tent","cubic","sine")

CFG=Config()
if SMOKE_TEST:
    CFG.steps_A=40; CFG.steps_B=30; CFG.seeds=2; CFG.eval_n=40
    CFG.calib_steps=30; CFG.calib_warm=10; CFG.benettin_warmup=10; CFG.benettin_steps=30; CFG.benettin_traj=4
    print("*** SMOKE TEST ***")
print(json.dumps({k:v for k,v in asdict(CFG).items()},indent=2,default=str))

{
  "state_dim": 64,
  "d_model": 64,
  "n_layers": 1,
  "mlp_mult": 2,
  "steps_A": 3000,
  "steps_B": 2500,
  "batch": 64,
  "warmup_frac": 0.05,
  "lr_min_frac": 0.1,
  "grad_clip": 1.0,
  "lr": 0.001,
  "seeds": 3,
  "eval_n": 1200,
  "eval_seed": 12345,
  "copy_len": 12,
  "vocab": 16,
  "fc_length": 64,
  "fc_bins": 32,
  "lambda_target": {
    "edge": -0.02,
    "deep": 0.3
  },
  "lambda_tol": 0.06,
  "calib_steps": 150,
  "calib_warm": 50,
  "benettin_warmup": 80,
  "benettin_steps": 300,
  "benettin_traj": 10,
  "map_families": [
    "logistic",
    "tent",
    "cubic",
    "sine"
  ]
}


## 1. Four chaotic local maps, one shared cell architecture

Each map takes a bounded state $h \in [0,1]^N$ and a scalar-ish rate/gain and returns an unbounded
pre-fold value; the SAME `fold` and SAME gain calibration wrap all four, so only the local
nonlinearity's functional form differs.

- logistic: $r\,h(1-h)$
- tent:     $r\cdot\min(h,\,1-h)$ (natively piecewise-linear; classic tent map generalization)
- cubic:    $r\,h(1-h^2)$ (cubic / Chebyshev-family nonlinearity, different bifurcation route)
- sine:     $r\cdot\frac{1}{\pi}\sin(\pi h)$ (smooth, bounded, structurally unrelated to logistic)

In [3]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

def local_map(h, r, family):
    if family=="logistic": return r*h*(1-h)
    if family=="tent":     return r*torch.minimum(h, 1-h)
    if family=="cubic":    return r*h*(1-h*h)
    if family=="sine":     return r*torch.sin(math.pi*h)/math.pi
    raise ValueError(family)

class ChaoticMap(nn.Module):
    def __init__(self,N,family,r_cap=4.0,gi=1.0,learnable_gamma=True):
        super().__init__(); self.family=family; self.r_cap=float(r_cap)
        self.w_r=nn.Parameter(torch.zeros(N))
        W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        wg=torch.tensor(float(math.log(math.expm1(gi))))
        self.w_gamma=nn.Parameter(wg) if learnable_gamma else wg
        self.learnable_gamma=learnable_gamma
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad():
            new=torch.tensor(float(math.log(math.expm1(max(val,1e-3)))))
            if self.learnable_gamma: self.w_gamma.data.copy_(new)
            else: self.w_gamma=new
    def forward(self,h):
        r=self.get_r(); u=h@self.W_mix.T+local_map(h,r,self.family)
        return triangle_fold(self.get_gamma()*u)

## 2. Cells: gated (experiment A) and frozen-reservoir (experiment B)

In [4]:
class GatedChaosCell(nn.Module):
    def __init__(self,cfg,family):
        super().__init__(); self.N=cfg.state_dim
        self.phi=ChaoticMap(cfg.state_dim,family,r_cap=4.0,gi=1.0,learnable_gamma=True)
        self.A=nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9,1.1)))
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.W_g=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.b_g=nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self,h,x):
        g=torch.sigmoid(x@self.W_g.T+self.b_g)
        raw=g*self.phi(h@self.A.T)+(1-g)*h+x@self.B.T
        return triangle_fold(raw)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)

class ReservoirChaosCell(nn.Module):
    # No gate (fair-test style); B may be frozen (canonical RC) via freeze_input().
    def __init__(self,cfg,family,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime
        self.phi=ChaoticMap(cfg.state_dim,family,r_cap=4.0,gi=1.0,learnable_gamma=True)
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.phi.w_r,self.phi.W_mix,self.phi.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCellRes(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

## 3. Tasks, gamma-calibration, and the validated Benettin/QR estimator

In [5]:
V=CFG.vocab; SEP,PAD=V,V+1; VOCAB_SIZE=V+2
def make_copy(cfg,batch,device,gen):
    L=cfg.copy_len; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1)
    return inp,tgt,"tokens",VOCAB_SIZE,VOCAB_SIZE
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt,"scalar",1,cfg.fc_bins
TASK_FN=dict(copy=make_copy,forecast=make_forecast)
@torch.no_grad()
def majority_class_chance(cfg,task,n=15000,seed=777):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(seed)
    _,tgt,_,_,_=TASK_FN[task](cfg,n,DEVICE,g); m=tgt!=-100
    nb=cfg.fc_bins if task=="forecast" else cfg.vocab
    counts=torch.bincount(tgt[m].reshape(-1),minlength=nb)
    return float(counts.max())/float(counts.sum())
CHANCE={t: majority_class_chance(CFG,t) for t in ("copy","forecast")}
print("chance:",{k:round(v,4) for k,v in CHANCE.items()})

def build_eval(cfg,task):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return TASK_FN[task](cfg,cfg.eval_n,DEVICE,g)
@torch.no_grad()
def eval_acc(model,evalset,chunk=256):
    inp,tgt,_,_,_=evalset; model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); tg=tgt[i:i+chunk]; m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0):
    steps=cfg.calib_steps; warm=cfg.calib_warm; N=cfg.state_dim; dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z): return triangle_fold(cell.get_gamma()*(z@cell.W_mix.T+local_map(z,cell.get_r(),cell.family)))
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0,1)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    target=cfg.lambda_target[regime]; lo,hi=0.05,10.0; best=None; best_err=1e9
    for _ in range(16):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1]

def logistic_lyap(r=4.0,n=3000,warm=200,seed=0):
    rng=np.random.default_rng(seed); x=rng.uniform(0.1,0.9); s=0.0; c=0
    for t in range(n+warm):
        d=abs(r*(1-2*x)); x=r*x*(1-x)
        if t>=warm: s+=math.log(d+1e-12); c+=1
    return s/max(c,1)
KNOWN=logistic_lyap(); print(f"[validation] logistic r=4 lambda={KNOWN:.4f} vs ln2={math.log(2):.4f} err={abs(KNOWN-math.log(2)):.4f}")

chance: {'copy': 0.0632, 'forecast': 0.1028}
[validation] logistic r=4 lambda=0.6927 vs ln2=0.6931 err=0.0004


## 4. Conditional-lambda estimator (Benettin/QR, real inputs)

In [6]:
@torch.no_grad()
def real_input_sequence(model,cfg,seed,layer,length,task,in_kind,is_reservoir):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(500+seed)
    inp,_,_,_,_=TASK_FN[task](cfg,1,DEVICE,g)
    e = model.emb(inp) if in_kind=="tokens" else model.inp(inp.unsqueeze(-1))
    dt=e.dtype; T=inp.shape[1]
    if is_reservoir:
        cell=model.cell; h=cell.init_state(1,DEVICE,dt); cap=[]
        for t in range(T):
            u=e[:,t]; cap.append(u.squeeze(0).detach()); h=cell.step(h,u)
    else:
        blk=model.blocks[layer]; h=blk.init_state(1,DEVICE,dt); cap=[]
        for t in range(T):
            u=blk.ln1(e[:,t]); cap.append(u.squeeze(0).detach())
            hh=blk.cell.step(h,u); y=e[:,t]+hh; y=y+blk.mlp(blk.ln2(y)); h=hh; e[:,t]=y if False else e[:,t]
        # (kept minimal: gated-model conditional lambda not needed for family-generality claim beyond A)
    seq=torch.stack(cap,0)
    if seq.shape[0]<length: seq=seq.repeat((length//seq.shape[0])+1,1)
    return seq[:length]

@torch.no_grad()
def conditional_lambda_reservoir(model,cfg,seed,task,in_kind):
    cell=model.cell; N=cfg.state_dim; total=cfg.benettin_warmup+cfg.benettin_steps
    lam_maxes=[]
    for tr in range(cfg.benettin_traj):
        g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(9000+seed*97+tr)
        seqx=real_input_sequence(model,cfg,9000+seed*97+tr,0,total,task,in_kind,True)
        h=torch.full((N,),0.5,device=DEVICE)
        Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=DEVICE))[0]; logsum=torch.zeros(N,device=DEVICE)
        for t in range(total):
            x=seqx[t%seqx.shape[0]]
            J=jacrev(lambda z: cell.step(z.unsqueeze(0),x.unsqueeze(0)).squeeze(0))(h)
            h=cell.step(h.unsqueeze(0),x.unsqueeze(0)).squeeze(0)
            Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
            if t>=cfg.benettin_warmup: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
        lam_maxes.append(float(torch.sort(logsum/cfg.benettin_steps,descending=True)[0][0]))
    return float(np.mean(lam_maxes))

## 5. Experiment A -- suppression under end-to-end training (gated cell, all 4 map families, copy task)

In [7]:
class GatedLM(nn.Module):
    def __init__(self,cfg,family,in_dim,out_dim):
        super().__init__()
        self.emb=nn.Embedding(in_dim,cfg.d_model)
        self.blocks=nn.ModuleList([_Blk(cfg,family)])
        self.ln_f=nn.LayerNorm(cfg.d_model); self.readout=nn.Linear(cfg.d_model,out_dim)
    def forward(self,x):
        e=self.emb(x); B,T=e.shape[0],e.shape[1]; dt=e.dtype
        st=[blk.init_state(B,e.device,dt) for blk in self.blocks]; outs=[]
        for t in range(T):
            xt=e[:,t]
            for li,blk in enumerate(self.blocks): st[li],xt=blk.step(st[li],xt)
            outs.append(self.readout(self.ln_f(xt)))
        return torch.stack(outs,1)

class _Blk(nn.Module):
    def __init__(self,cfg,family):
        super().__init__(); self.ln1=nn.LayerNorm(cfg.d_model); self.cell=GatedChaosCell(cfg,family)
        self.ln2=nn.LayerNorm(cfg.d_model); h=cfg.d_model*cfg.mlp_mult
        self.mlp=nn.Sequential(nn.Linear(cfg.d_model,h),nn.GELU(),nn.Linear(h,cfg.d_model))
    def step(self,hs,x):
        u=self.ln1(x); hs=self.cell.step(hs,u); y=x+hs; y=y+self.mlp(self.ln2(y)); return hs,y
    def init_state(self,b,dev,dt): return self.cell.init_state(b,dev,dt)

def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

@torch.no_grad()
def conditional_lambda_gated(model,cfg,seed,task):
    blk=model.blocks[0]; cell=blk.cell; N=cfg.state_dim; total=cfg.benettin_warmup+cfg.benettin_steps
    lam_maxes=[]
    for tr in range(cfg.benettin_traj):
        g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(9000+seed*97+tr)
        gg=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gg.manual_seed(500+seed*3+tr)
        inp,_,_,_,_=TASK_FN[task](cfg,1,DEVICE,gg); e=model.emb(inp)
        seqx=blk.ln1(e[0])
        h=torch.full((N,),0.5,device=DEVICE)
        Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=DEVICE))[0]; logsum=torch.zeros(N,device=DEVICE)
        for t in range(total):
            x=seqx[t%seqx.shape[0]]
            J=jacrev(lambda z: cell.step(z.unsqueeze(0),x.unsqueeze(0)).squeeze(0))(h)
            h=cell.step(h.unsqueeze(0),x.unsqueeze(0)).squeeze(0)
            Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
            if t>=cfg.benettin_warmup: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
        lam_maxes.append(float(torch.sort(logsum/cfg.benettin_steps,descending=True)[0][0]))
    return float(np.mean(lam_maxes))

rowsA=[]; t0=time.time()
for family in CFG.map_families:
    for sd in range(CFG.seeds):
        torch.manual_seed(sd); np.random.seed(sd)
        gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(sd)
        model=GatedLM(CFG,family,VOCAB_SIZE,VOCAB_SIZE).to(DEVICE)
        opt=torch.optim.AdamW(model.parameters(),lr=CFG.lr,betas=(0.9,0.95))
        sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(CFG,CFG.steps_A))
        div=False
        for step in range(CFG.steps_A):
            inp,tgt,_,_,_=TASK_FN["copy"](CFG,CFG.batch,DEVICE,gen)
            lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
            opt.zero_grad(set_to_none=True); L.backward()
            gn=torch.nn.utils.clip_grad_norm_(model.parameters(),CFG.grad_clip).item(); opt.step(); sched.step()
            if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
        acc=eval_acc(model,build_eval(CFG,"copy")) if not div else float("nan")
        lam=conditional_lambda_gated(model,CFG,sd,"copy") if not div else float("nan")
        rowsA.append(dict(family=family,seed=sd,diverged=div,acc=acc,conditional_lambda=lam))
        print(f"[A] {family:9s} s={sd} acc={acc:.4f} lam={lam:+.3f}" + (" DIV" if div else ""))
RESA=pd.DataFrame(rowsA); savetable(RESA,"phase13_expA_suppression.csv")
print(f"experiment A wall clock: {(time.time()-t0)/60:.1f} min")

[A] logistic  s=0 acc=0.0625 lam=+1.310
[A] logistic  s=1 acc=0.0609 lam=+1.290
[A] logistic  s=2 acc=0.0626 lam=+1.218
[A] tent      s=0 acc=0.0699 lam=+0.419
[A] tent      s=1 acc=0.0633 lam=+0.339
[A] tent      s=2 acc=0.0830 lam=+0.339
[A] cubic     s=0 acc=nan lam=+nan DIV
[A] cubic     s=1 acc=nan lam=+nan DIV
[A] cubic     s=2 acc=nan lam=+nan DIV
[A] sine      s=0 acc=0.3940 lam=-0.050
[A] sine      s=1 acc=0.3835 lam=-0.038
[A] sine      s=2 acc=0.4151 lam=-0.045
saved /content/drive/My Drive/ChaoSSM/results/phase13_map_generality/phase13_expA_suppression.csv
experiment A wall clock: 38.8 min


## 6. Experiment B -- benefit under the canonical frozen reservoir protocol (all 4 families, forecast task)

In [8]:
class ReservoirLM(nn.Module):
    def __init__(self,cfg,cell,out_dim):
        super().__init__(); self.inp=nn.Linear(1,cfg.d_model); self.cell=cell
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
    def forward(self,x):
        e=self.inp(x.unsqueeze(-1)); B,T=e.shape[0],e.shape[1]; dt=e.dtype
        h=self.cell.init_state(B,e.device,dt); outs=[]
        for t in range(T):
            h=self.cell.step(h,e[:,t]); outs.append(self.readout(self.ln(h)))
        return torch.stack(outs,1)

rowsB=[]; t0=time.time()
for family in CFG.map_families:
    for regime in ("edge",):   # edge is the load-bearing regime per Phase 10; keep B cheap
        for sd in range(CFG.seeds):
            torch.manual_seed(sd); np.random.seed(sd)
            gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(sd)
            cell=ReservoirChaosCell(CFG,family,regime).to(DEVICE)
            gam,lam_cal=calibrate_gamma(cell.phi,CFG,regime,seed=sd)
            cell.freeze_recurrence(); cell.freeze_input()
            model=ReservoirLM(CFG,cell,CFG.fc_bins).to(DEVICE)
            params=[p for p in model.parameters() if p.requires_grad]
            opt=torch.optim.AdamW(params,lr=CFG.lr,betas=(0.9,0.95))
            sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(CFG,CFG.steps_B))
            div=False
            for step in range(CFG.steps_B):
                inp,tgt,_,_,_=TASK_FN["forecast"](CFG,CFG.batch,DEVICE,gen)
                lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1))
                opt.zero_grad(set_to_none=True); L.backward()
                gn=torch.nn.utils.clip_grad_norm_(params,CFG.grad_clip).item(); opt.step(); sched.step()
                if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
            acc=eval_acc(model,build_eval(CFG,"forecast")) if not div else float("nan")
            rowsB.append(dict(family=family,regime=regime,arm="chaos",seed=sd,diverged=div,acc=acc,calibrated_lambda=lam_cal))
            print(f"[B] {family:9s} chaos-{regime} s={sd} acc={acc:.4f} lam_cal={lam_cal:+.3f}" + (" DIV" if div else ""))
    # matched linear reservoir (frozen recurrence + frozen input), per family iteration for a shared table
    for sd in range(CFG.seeds):
        torch.manual_seed(1000+sd); np.random.seed(1000+sd)
        gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(1000+sd)
        lincell=LinearCellRes(CFG).to(DEVICE); lincell.freeze_recurrence(); lincell.freeze_input()
        model=ReservoirLM(CFG,lincell,CFG.fc_bins).to(DEVICE)
        params=[p for p in model.parameters() if p.requires_grad]
        opt=torch.optim.AdamW(params,lr=CFG.lr,betas=(0.9,0.95))
        sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(CFG,CFG.steps_B))
        div=False
        for step in range(CFG.steps_B):
            inp,tgt,_,_,_=TASK_FN["forecast"](CFG,CFG.batch,DEVICE,gen)
            lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1))
            opt.zero_grad(set_to_none=True); L.backward()
            gn=torch.nn.utils.clip_grad_norm_(params,CFG.grad_clip).item(); opt.step(); sched.step()
            if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
        acc=eval_acc(model,build_eval(CFG,"forecast")) if not div else float("nan")
        rowsB.append(dict(family=family,regime="linear-control",arm="linear",seed=sd,diverged=div,acc=acc,calibrated_lambda=float("nan")))
        print(f"[B] {family:9s} linear      s={sd} acc={acc:.4f}" + (" DIV" if div else ""))
RESB=pd.DataFrame(rowsB); savetable(RESB,"phase13_expB_benefit.csv")
print(f"experiment B wall clock: {(time.time()-t0)/60:.1f} min")

[B] logistic  chaos-edge s=0 acc=0.6272 lam_cal=-0.022
[B] logistic  chaos-edge s=1 acc=0.6748 lam_cal=-0.026
[B] logistic  chaos-edge s=2 acc=0.6513 lam_cal=-0.029
[B] logistic  linear      s=0 acc=0.3202
[B] logistic  linear      s=1 acc=0.3157
[B] logistic  linear      s=2 acc=0.3277
[B] tent      chaos-edge s=0 acc=0.3868 lam_cal=+0.026
[B] tent      chaos-edge s=1 acc=0.4155 lam_cal=+0.024
[B] tent      chaos-edge s=2 acc=0.3946 lam_cal=+0.021
[B] tent      linear      s=0 acc=0.3202
[B] tent      linear      s=1 acc=0.3157
[B] tent      linear      s=2 acc=0.3277
[B] cubic     chaos-edge s=0 acc=0.4507 lam_cal=+0.020
[B] cubic     chaos-edge s=1 acc=0.2920 lam_cal=+0.017
[B] cubic     chaos-edge s=2 acc=0.0333 lam_cal=-0.041
[B] cubic     linear      s=0 acc=0.3202
[B] cubic     linear      s=1 acc=0.3157
[B] cubic     linear      s=2 acc=0.3277
[B] sine      chaos-edge s=0 acc=0.6112 lam_cal=+0.018
[B] sine      chaos-edge s=1 acc=0.6381 lam_cal=+0.014
[B] sine      chaos-edge s

## 7. Analysis + figure

In [9]:
sumA=RESA.groupby("family").agg(acc_mean=("acc","mean"),lam_mean=("conditional_lambda","mean")).round(4).reset_index()
print("Experiment A (suppression under end-to-end training, copy):"); print(sumA.to_string(index=False))

pivB=RESB.pivot_table(index="family",columns="arm",values="acc",aggfunc="mean").round(4)
print("\nExperiment B (frozen canonical reservoir, forecast, chaos-edge vs linear):"); print(pivB.to_string())
savetable(sumA,"phase13_summary_A.csv"); savetable(pivB.reset_index(),"phase13_summary_B.csv")

fig,axes=plt.subplots(1,2,figsize=(12.5,4.6))
axes[0].bar(sumA.family,sumA.lam_mean,color=["tab:green" if v<0 else "tab:red" for v in sumA.lam_mean])
axes[0].axhline(0,color="k",lw=1); axes[0].set_ylabel("conditional lambda (trained, copy)")
axes[0].set_title("Exp A: suppression by map family"); axes[0].grid(alpha=.3,axis="y")
x=np.arange(len(pivB)); w=0.35
axes[1].bar(x-w/2,pivB["chaos"],w,label="chaos-edge (frozen, random input)",color="tab:green")
axes[1].bar(x+w/2,pivB["linear"],w,label="linear (frozen, random input)",color="tab:blue")
axes[1].set_xticks(x); axes[1].set_xticklabels(pivB.index); axes[1].set_ylabel("forecast accuracy")
axes[1].set_title("Exp B: canonical-RC benefit by map family"); axes[1].legend(fontsize=8); axes[1].grid(alpha=.3,axis="y")
fig.suptitle("Phase 13: generality across chaotic map families")
fig.tight_layout(); savefig(fig,"phase13_generality_bars.png"); plt.close(fig)

Experiment A (suppression under end-to-end training, copy):
  family  acc_mean  lam_mean
   cubic       NaN       NaN
logistic    0.0620    1.2726
    sine    0.3976   -0.0445
    tent    0.0720    0.3658

Experiment B (frozen canonical reservoir, forecast, chaos-edge vs linear):
arm        chaos  linear
family                  
cubic     0.2587  0.3212
logistic  0.6511  0.3212
sine      0.4333  0.3212
tent      0.3990  0.3212
saved /content/drive/My Drive/ChaoSSM/results/phase13_map_generality/phase13_summary_A.csv
saved /content/drive/My Drive/ChaoSSM/results/phase13_map_generality/phase13_summary_B.csv
saved /content/drive/My Drive/ChaoSSM/results/phase13_map_generality/phase13_generality_bars.png


## 8. Verdict

In [10]:
supp_holds = bool((sumA.lam_mean<0).all())
benefit_holds = bool((pivB["chaos"]>pivB["linear"]).all())
n_supp = int((sumA.lam_mean<0).sum()); n_ben = int((pivB["chaos"]>pivB["linear"]).sum())

if supp_holds and benefit_holds:
    decision=(f"GENERALITY CONFIRMED ACROSS ALL {len(CFG.map_families)} MAP FAMILIES. Suppression under "
              f"end-to-end training holds for {n_supp}/{len(CFG.map_families)} families (conditional "
              f"lambda<0 for all: {dict(zip(sumA.family,sumA.lam_mean.round(3)))}), and the canonical-RC "
              f"benefit holds for {n_ben}/{len(CFG.map_families)} families ({pivB.round(3).to_dict()}). "
              "Both headline findings are properties of trained/frozen chaotic recurrence in general, "
              "not artifacts of the logistic-fold map's specific construction. Report this as a "
              "generality check in the paper's Discussion or an appendix.")
else:
    decision=(f"PARTIAL GENERALITY. Suppression held for {n_supp}/{len(CFG.map_families)} families; "
              f"canonical-RC benefit held for {n_ben}/{len(CFG.map_families)} families. Report exactly "
              "which map families depart from the logistic-map pattern, and inspect whether the "
              "departure tracks a specific structural feature (e.g. smoothness, symmetry) of that map.")

verdict=dict(smoke_test=SMOKE_TEST, families=list(CFG.map_families),
             expA_conditional_lambda=dict(zip(sumA.family,sumA.lam_mean)),
             expB_chaos_vs_linear=pivB.to_dict(),
             suppression_holds_all=supp_holds, benefit_holds_all=benefit_holds,
             decision=decision)
savejson(verdict,"phase13_verdict.json")
print(decision); print(); print(json.dumps(verdict,indent=2,default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase13_map_generality/phase13_verdict.json
PARTIAL GENERALITY. Suppression held for 1/4 families; canonical-RC benefit held for 3/4 families. Report exactly which map families depart from the logistic-map pattern, and inspect whether the departure tracks a specific structural feature (e.g. smoothness, symmetry) of that map.

{
  "smoke_test": false,
  "families": [
    "logistic",
    "tent",
    "cubic",
    "sine"
  ],
  "expA_conditional_lambda": {
    "cubic": NaN,
    "logistic": 1.2726,
    "sine": -0.0445,
    "tent": 0.3658
  },
  "expB_chaos_vs_linear": {
    "chaos": {
      "cubic": 0.2587,
      "logistic": 0.6511,
      "sine": 0.4333,
      "tent": 0.399
    },
    "linear": {
      "cubic": 0.3212,
      "logistic": 0.3212,
      "sine": 0.3212,
      "tent": 0.3212
    }
  },
  "suppression_holds_all": false,
  "benefit_holds_all": false,
  "decision": "PARTIAL GENERALITY. Suppression held for 1/4 families; canonical-RC ben